# Consultas SQL — Dataset de Clasificación: Diabetes Health Indicators

Este notebook reproduce las siete consultas SQL definidas en `src/consultas_clasificacion.py`,
añadiendo explicaciones sobre la motivación analítica de cada una y el esquema de la base de datos.

**Requisito previo:** haber ejecutado `src/crear_base_datos.py` para generar
`database/diabetes_clasificacion.db`.

---

**Dataset:** BRFSS 2015 — CDC (Center for Disease Control and Prevention)
**Tabla principal:** `diabetes` (253 680 filas × 22 columnas)
**Tabla de metadatos:** `variables_metadata` (nombre, descripción y tipo de cada variable)
**Variable objetivo:** `Diabetes_binary` — 0 = sin diabetes / 1 = prediabetes o diabetes

## 1. Configuración: conexión a la base de datos

Se utiliza `pathlib` para construir la ruta de forma robusta independientemente
desde dónde se ejecute el notebook.
`sqlite3` es el motor integrado de Python para bases de datos SQLite, y `pandas`
permite mostrar los resultados en formato tabular dentro del notebook.

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

# Resuelve la ruta a la BD sin importar el directorio de trabajo actual
_notebook_dir = Path(os.path.abspath('')) if '__file__' not in dir() else Path(__file__).parent
import os
_notebook_dir = Path(os.path.abspath(''))

# El notebook vive en notebooks/, la BD en database/ al mismo nivel
PROJECT_DIR = _notebook_dir.parent if _notebook_dir.name == 'notebooks' else _notebook_dir
DB_PATH = PROJECT_DIR / "database" / "diabetes_clasificacion.db"

conn = sqlite3.connect(DB_PATH)
print(f"Conectado a: {DB_PATH}")
print(f"Archivo existe: {DB_PATH.exists()}")

---
## 2. Consulta 1 — Distribución de la variable objetivo

**Objetivo analítico:** Cuantificar el desbalanceo de clases en el dataset.

El desbalanceo es el eje central del Laboratorio 5:
- Clase **0** (sin diabetes): ~86 % de los registros
- Clase **1** (prediabetes/diabetes): ~14 % de los registros

Un clasificador que prediga siempre `0` obtendría 86 % de accuracy sin aprender nada.
Por eso se priorizan métricas como **Recall** y **F1** sobre la exactitud global.

**Cláusulas clave:**
- `COUNT(*) * 100.0 / (SELECT COUNT(*) FROM diabetes)` — subconsulta escalar para calcular el porcentaje
- `GROUP BY Diabetes_binary` — agrupa por valor de la variable objetivo (0 y 1)
- `ORDER BY Diabetes_binary` — ordena clase 0 primero

In [ ]:
q1 = """
SELECT
    Diabetes_binary                                          AS clase,
    COUNT(*)                                                 AS cantidad,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM diabetes), 2) AS porcentaje_pct
FROM diabetes
GROUP BY Diabetes_binary
ORDER BY Diabetes_binary;
"""

print("Distribución de la variable objetivo:")
display(pd.read_sql_query(q1, conn))

---
## 3. Consulta 2 — Promedios de variables clave por clase

**Objetivo analítico:** Comparar el perfil promedio de los grupos con y sin diabetes
para identificar qué variables difieren más entre clases.

Variables seleccionadas por su relevancia clínica y correlación conocida con la diabetes:

| Variable | Descripción |
|---|---|
| `BMI` | Índice de Masa Corporal (obesidad) |
| `Age` | Grupo de edad codificado 1–13 |
| `GenHlth` | Salud general autopercibida 1–5 (1 = excelente) |
| `MentHlth` | Días de mala salud mental en los últimos 30 días |
| `PhysHlth` | Días de mala salud física en los últimos 30 días |
| `Income` | Nivel de ingresos codificado 1–8 |

**Cláusulas clave:**
- `AVG()` + `ROUND(..., 2)` — media con 2 decimales por grupo
- `GROUP BY Diabetes_binary` — un promedio por clase

In [ ]:
q2 = """
SELECT
    Diabetes_binary                  AS clase,
    ROUND(AVG(BMI), 2)               AS bmi_promedio,
    ROUND(AVG(Age), 2)               AS edad_grupo_prom,
    ROUND(AVG(GenHlth), 2)           AS salud_general_prom,
    ROUND(AVG(MentHlth), 2)          AS dias_salud_mental_prom,
    ROUND(AVG(PhysHlth), 2)          AS dias_salud_fisica_prom,
    ROUND(AVG(Income), 2)            AS ingresos_prom
FROM diabetes
GROUP BY Diabetes_binary
ORDER BY Diabetes_binary;
"""

print("Promedios de variables clave por clase:")
display(pd.read_sql_query(q2, conn))

---
## 4. Consulta 3 — Tasa de diabetes por grupo de edad

**Objetivo analítico:** Observar cómo aumenta la prevalencia de diabetes con la edad.

`Age` está codificada como variable ordinal discreta (1–13):

| Código | Rango de edad |
|---|---|
| 1 | 18–24 |
| 2 | 25–29 |
| ... | ... |
| 9 | 60–64 |
| 13 | 80+ |

**Cláusulas clave:**
- `SUM(Diabetes_binary)` — cuenta los positivos (clase 1) ya que la variable es 0/1
- `SUM(...) * 100.0 / COUNT(*)` — calcula la tasa porcentual por grupo de edad
- `ORDER BY Age` — ordena de menor a mayor grupo de edad para ver la tendencia creciente

In [ ]:
q3 = """
SELECT
    Age                                                         AS grupo_edad,
    COUNT(*)                                                    AS total,
    SUM(Diabetes_binary)                                        AS con_diabetes,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2)          AS tasa_diabetes_pct
FROM diabetes
GROUP BY Age
ORDER BY Age;
"""

print("Tasa de diabetes por grupo de edad:")
display(pd.read_sql_query(q3, conn))

---
## 5. Consulta 4 — Personas no diabéticas con múltiples factores de riesgo

**Objetivo analítico:** Identificar el volumen de personas que aún no tienen diagnóstico
de diabetes (`Diabetes_binary = 0`) pero presentan **5 factores de riesgo simultáneos**:

| Factor | Condición |
|---|---|
| `HighBP = 1` | Hipertensión arterial |
| `HighChol = 1` | Colesterol alto |
| `BMI >= 30` | Obesidad (IMC ≥ 30) |
| `PhysActivity = 0` | Sedentarismo |
| `Smoker = 1` | Fumador |

Este subgrupo es de alto interés clínico: son personas en riesgo que aún pueden ser
intervenidas preventivamente. Un modelo de ML con buen Recall debería alertar sobre ellos.

**Cláusulas clave:**
- `WHERE ... AND ... AND ...` — filtro compuesto con cinco condiciones simultáneas
- `COUNT(*)` — cuenta el número de registros que cumplen todas las condiciones

In [ ]:
q4 = """
SELECT
    COUNT(*) AS no_diabeticos_alto_riesgo
FROM diabetes
WHERE Diabetes_binary  = 0
  AND HighBP           = 1
  AND HighChol         = 1
  AND BMI             >= 30
  AND PhysActivity     = 0
  AND Smoker           = 1;
"""

resultado_q4 = pd.read_sql_query(q4, conn)
total = resultado_q4["no_diabeticos_alto_riesgo"].iloc[0]
print(f"Personas no diabéticas con 5 factores de riesgo simultáneos: {total:,}")
display(resultado_q4)

---
## 6. Consulta 5 — Tasa de diabetes por nivel educativo e ingresos

**Objetivo analítico:** Explorar la relación sociodemográfica con la diabetes.
La literatura epidemiológica documenta que menores ingresos y menor educación
se asocian con mayor prevalencia de enfermedades crónicas.

Codificaciones:

| Variable | Escala |
|---|---|
| `Education` | 1 = Nunca fue a la escuela … 6 = Universitario |
| `Income` | 1 = < $10k … 8 = > $75k |

**Cláusulas clave:**
- `GROUP BY Education, Income` — agrupación doble para crear una tabla cruzada
- `ORDER BY Education, Income` — ordena de menor a mayor nivel en ambas dimensiones
- `.head(20)` en pandas — la consulta puede devolver hasta 48 combinaciones (6×8);
  se trunca a 20 para visualización rápida

In [ ]:
q5 = """
SELECT
    Education                                              AS nivel_educativo,
    Income                                                 AS nivel_ingresos,
    COUNT(*)                                               AS total,
    SUM(Diabetes_binary)                                   AS con_diabetes,
    ROUND(SUM(Diabetes_binary) * 100.0 / COUNT(*), 2)     AS tasa_pct
FROM diabetes
GROUP BY Education, Income
ORDER BY Education, Income;
"""

df_q5 = pd.read_sql_query(q5, conn)
print(f"Total de combinaciones Education × Income: {len(df_q5)}")
print("\nPrimeras 20 filas:")
display(df_q5.head(20))

---
## 7. Consulta 6 — JOIN: prevalencia de factores de riesgo con metadatos

**Objetivo analítico:** Combinar la tabla `diabetes` (datos) con la tabla
`variables_metadata` (descripciones) para enriquecer el reporte de prevalencia
de factores de riesgo binarios con su nombre legible y tipo.

**Diseño de la consulta:**
- `JOIN diabetes d ON 1=1` — join cartesiano con una sola fila de metadata por variable;
  se necesita para que `AVG(CASE ...)` opere sobre toda la tabla `diabetes`
- `CASE m.variable WHEN ... THEN d.<columna>` — selección dinámica de la columna
  de datos según el nombre de la variable en metadata; evita 7 subconsultas separadas
- `WHERE m.tipo = 'Categórica nominal'` — filtra solo variables binarias de factores de riesgo
- `WHERE m.variable NOT IN (...)` — excluye la variable objetivo y variables de estilo de vida
  no directamente asociadas a factores de riesgo cardiovascular/metabólico

**Resultado:** ranking de factores de riesgo por prevalencia poblacional

In [ ]:
q6 = """
SELECT
    m.variable,
    m.descripcion,
    m.tipo,
    ROUND(AVG(
        CASE m.variable
            WHEN 'HighBP'              THEN d.HighBP
            WHEN 'HighChol'            THEN d.HighChol
            WHEN 'Smoker'              THEN d.Smoker
            WHEN 'Stroke'              THEN d.Stroke
            WHEN 'HeartDiseaseorAttack' THEN d.HeartDiseaseorAttack
            WHEN 'PhysActivity'        THEN d.PhysActivity
            WHEN 'DiffWalk'            THEN d.DiffWalk
        END
    ) * 100, 2) AS prevalencia_pct
FROM variables_metadata m
JOIN diabetes d ON 1=1
WHERE m.tipo = 'Categorica nominal'
  AND m.variable NOT IN ('Diabetes_binary', 'Sex', 'CholCheck',
                          'Fruits', 'Veggies', 'HvyAlcoholConsump',
                          'AnyHealthcare', 'NoDocbcCost')
GROUP BY m.variable, m.descripcion, m.tipo
ORDER BY prevalencia_pct DESC;
"""

print("Prevalencia de factores de riesgo (orden descendente):")
display(pd.read_sql_query(q6, conn))

---
## 8. Consulta 7 — IMC promedio por sexo y clase de diabetes

**Objetivo analítico:** Analizar si la relación entre IMC y diabetes difiere
según el sexo del encuestado.

`Sex` está codificada como:
- `0` = Mujer
- `1` = Hombre

**Estadísticos reportados:** promedio, mínimo y máximo de BMI para cada combinación
(sexo × clase), lo que permite visualizar no solo la tendencia central sino también
la dispersión de los valores de IMC.

**Cláusulas clave:**
- `GROUP BY Sex, Diabetes_binary` — segmenta en cuatro grupos: {mujer sin diabetes,
  mujer con diabetes, hombre sin diabetes, hombre con diabetes}
- `AVG`, `MIN`, `MAX` — tres funciones de agregación sobre BMI en una sola consulta
- `ORDER BY Sex, Diabetes_binary` — ordena por sexo y dentro de cada sexo por clase

In [ ]:
q7 = """
SELECT
    Sex                              AS sexo,
    Diabetes_binary                  AS clase,
    COUNT(*)                         AS total,
    ROUND(AVG(BMI), 2)               AS bmi_promedio,
    ROUND(MIN(BMI), 2)               AS bmi_min,
    ROUND(MAX(BMI), 2)               AS bmi_max
FROM diabetes
GROUP BY Sex, Diabetes_binary
ORDER BY Sex, Diabetes_binary;
"""

print("IMC (BMI) por sexo y clase de diabetes:")
display(pd.read_sql_query(q7, conn))

---
## 9. Cierre de conexión

In [ ]:
conn.close()
print("Conexion a la base de datos cerrada correctamente.")
print("\nResumen de consultas ejecutadas:")
print("  1. Distribucion de la variable objetivo (desbalanceo de clases)")
print("  2. Promedios de variables clinicas por clase")
print("  3. Tasa de diabetes por grupo de edad")
print("  4. Conteo de no-diabeticos con 5 factores de riesgo simultaneos")
print("  5. Tasa de diabetes por nivel educativo e ingresos")
print("  6. JOIN con metadatos: prevalencia de factores de riesgo")
print("  7. IMC promedio por sexo y clase de diabetes")